# Feed forward neural network for nlp

In [1]:
import sys
import torch
import torch.nn as nn
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding

seed = 42
torch.manual_seed(seed)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:1000])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [8]:
text = text[:1000]
embedding_dim = 10

model = word_embedding(embedding_type="gpt2", embedding_dim=embedding_dim, seed=seed)
tokens = model.encode(text)
tokens = torch.tensor(tokens, dtype=torch.long)
vocab_size = len(set(tokens))
embedding = model.embed(tokens)

print("Number of tokens:", vocab_size)
print("Shape of embedding:", embedding)

Number of tokens: 31
Shape of embedding: Embedding(50257, 10)


In [9]:
# Split training and validation dataset
n = int(0.9*len(tokens)) # first 90% will be train, rest val
train_data = tokens[:n]
val_data = tokens[n:]

print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

train has 27 tokens
val has 4 tokens


In [10]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[ 502, 2740,   13,  198,  198, 3237,   25,  198],
        [  25,  198, 5248,  461,   11, 2740,   13,  198],
        [  11, 3285,  502, 2740,   13,  198,  198, 3237],
        [ 502, 2740,   13,  198,  198, 3237,   25,  198]])
targets:
torch.Size([4, 8])
tensor([[2740,   13,  198,  198, 3237,   25,  198, 5248],
        [ 198, 5248,  461,   11, 2740,   13,  198,  198],
        [3285,  502, 2740,   13,  198,  198, 3237,   25],
        [2740,   13,  198,  198, 3237,   25,  198, 5248]])


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)

IndexError: index out of range in self

In [21]:
from torch.nn import functional as F

def forward(embedding, idx, targets=None):

    # idx and targets are both (B,T) tensor of integers
    logits = embedding(idx) # (B,T,C)

    if targets is None:
        loss = None
    else:
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)

    return logits, loss

In [35]:
logits = embedding(xb)
B, T, C = logits.shape
print(B, T, C)

4 8 128


In [38]:
# B, T, C = logits.shape
# logits = logits.view(B*T, C)
# targets = targets.view(B*T)
loss = F.cross_entropy(xb, yb)

RuntimeError: Expected floating point type for target with class probabilities, got Long

In [32]:
embedding

Embedding(50257, 128)

In [31]:
yb

tensor([[30313,   262, 22397,   282,   290,   884,  3790,   198],
        [  438,   198, 10418,   329,   511, 11989,    11, 17743],
        [  322, 12105,   287,  3426,  6729,   198,  3886,  3612],
        [15581,  8636,    13,   198,   198, 35510,  4221,  5883]])

In [34]:
targets = yb.view(B*T)
targets

tensor([30313,   262, 22397,   282,   290,   884,  3790,   198,   438,   198,
        10418,   329,   511, 11989,    11, 17743,   322, 12105,   287,  3426,
         6729,   198,  3886,  3612, 15581,  8636,    13,   198,   198, 35510,
         4221,  5883])

In [ ]:
B, T, C = logits.shape

In [30]:
forward(embedding, xb, yb)

IndexError: Target 30313 is out of bounds.